In [ ]:
# ============================================================
# 交叉验证：用 Python 严谨引擎独立评估 MATLAB 解
# 对 MATLAB GA 输出的窗口 [5.09, 1.49, 0.10, 0.12]
# 做 MC200 大样本统计，检验其声称的 9.33% 是否成立
# ============================================================

import torch
import numpy as np
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ==========================================
# 系统参数 — 与 MATLAB search_channel.m 完全一致
# ==========================================
room_x, room_y, room_z_max = 10.0, 10.0, 3.0
x_BS, y_BS, z_BS = 10.0, -100.0, -10.0
E = 8; d_B = 0.075; lambda_val = 0.075; L1 = 2
d_ref = abs(y_BS) * 1.5; P_BS_dBm = 40.0; R_th = 0.2
N0_dBm_Hz = -174.0; B = 20e6; p_m = 1.0 / 5.0; n_users = 5
P_BS = 10**(P_BS_dBm / 10.0) * 1e-3
N0 = 10**(N0_dBm_Hz / 10.0) * 1e-3 * B

SIM_TIME = 300; DT = 10
TIME_STEPS = int(SIM_TIME / DT)
N_TOTAL_POINTS = n_users * TIME_STEPS  # 150
print(f'Trajectory: {n_users} users x {TIME_STEPS} steps = {N_TOTAL_POINTS} points')

# ==========================================
# 轨迹生成器 — 完整移植 MATLAB getHumanPosi_custom
# ==========================================
def generate_human_trajectory(rng=None):
    if rng is None: rng = np.random
    room_size = [0.0, room_x, 0.0, room_y]
    hotspot_center = np.array([room_x / 2, room_y / 2])
    hotspot_radius = room_x / 3.0
    p_t = 0.6; p_s = 0.3; tau_h = 1.5; tau_n = 0.1
    v_h = 0.2; v_n = 1.0; g_h = 0.6; g_n = 0.2
    time_steps = int(SIM_TIME / DT)

    def generate_target(current_pos):
        if rng.rand() < g_h:
            target = hotspot_center + (rng.rand(2) - 0.5) * 2 * hotspot_radius
        else:
            target = np.array([
                room_size[0] + rng.rand() * (room_size[1] - room_size[0]),
                room_size[2] + rng.rand() * (room_size[3] - room_size[2])])
        target[0] = np.clip(target[0], room_size[0], room_size[1])
        target[1] = np.clip(target[1], room_size[2], room_size[3])
        return target

    def get_tau(region):
        return 1.5 if region == 'hot' else 0.1
    def get_speed(region):
        return 0.2 if region == 'hot' else 1.0

    users_height = 1.5 + 0.5 * rng.rand(n_users)
    positions = np.zeros((n_users, time_steps, 2))
    user_pos = np.zeros((n_users, 2))
    user_region = [None] * n_users
    user_state = [None] * n_users
    user_target = np.zeros((n_users, 2))
    user_dir = np.zeros((n_users, 2))
    user_speed = np.zeros(n_users)
    user_pause_remain = np.zeros(n_users)

    for i in range(n_users):
        user_pos[i, 0] = room_size[0] + rng.rand() * (room_size[1] - room_size[0])
        user_pos[i, 1] = room_size[2] + rng.rand() * (room_size[3] - room_size[2])
        dist2hot = np.linalg.norm(user_pos[i] - hotspot_center)
        user_region[i] = 'hot' if dist2hot <= hotspot_radius else 'normal'
        if rng.rand() < p_t:
            user_state[i] = 'transfer'
            user_target[i] = generate_target(user_pos[i])
            dir_vec = user_target[i] - user_pos[i]
            user_dir[i] = dir_vec / np.linalg.norm(dir_vec)
            user_speed[i] = get_speed(user_region[i])
        else:
            user_state[i] = 'pause'
            user_pause_remain[i] = rng.exponential(get_tau(user_region[i]))
        positions[i, 0, :] = user_pos[i]

    for step in range(1, time_steps):
        for i in range(n_users):
            if user_state[i] == 'pause':
                user_pause_remain[i] -= DT
                positions[i, step, :] = user_pos[i]
                if user_pause_remain[i] <= 0:
                    user_state[i] = 'transfer'
                    user_target[i] = generate_target(user_pos[i])
                    dir_vec = user_target[i] - user_pos[i]
                    user_dir[i] = dir_vec / np.linalg.norm(dir_vec)
                    user_speed[i] = get_speed(user_region[i])
            else:
                move_dist = user_speed[i] * DT
                pos_diff = user_target[i] - user_pos[i]
                if np.linalg.norm(pos_diff) <= move_dist:
                    user_pos[i] = user_target[i].copy()
                    dist2hot = np.linalg.norm(user_pos[i] - hotspot_center)
                    user_region[i] = 'hot' if dist2hot <= hotspot_radius else 'normal'
                    if rng.rand() < p_s:
                        user_state[i] = 'pause'
                        user_pause_remain[i] = rng.exponential(get_tau(user_region[i]))
                    else:
                        user_target[i] = generate_target(user_pos[i])
                        dir_vec = user_target[i] - user_pos[i]
                        user_dir[i] = dir_vec / np.linalg.norm(dir_vec)
                        user_speed[i] = get_speed(user_region[i])
                else:
                    user_pos[i] = user_pos[i] + user_dir[i] * move_dist
                positions[i, step, :] = user_pos[i]

    total_points = np.zeros((n_users * time_steps, 3))
    idx = 0
    for user in range(n_users):
        for step in range(time_steps):
            total_points[idx, 0] = positions[user, step, 0]
            total_points[idx, 1] = positions[user, step, 1]
            total_points[idx, 2] = users_height[user]
            idx += 1
    return total_points

# ==========================================
# 信道模型 — 完整对齐 MATLAB equivalent_farfield_channel_2
# ==========================================
def equivalent_farfield_channel_pytorch(window_params, locs, nlos_params):
    if not isinstance(window_params, torch.Tensor):
        window_params = torch.tensor(window_params, dtype=torch.float32, device=device)
    xc, zc, Lx, Lz = window_params[0], window_params[1], window_params[2], window_params[3]
    xu, yu, zu = locs[:, 0], locs[:, 1], locs[:, 2]

    dx_BS = xc - x_BS; dy_BS = torch.tensor(0.0 - y_BS, device=device); dz_BS = zc - z_BS
    R_BW = torch.sqrt(dx_BS**2 + dy_BS**2 + dz_BS**2)
    theta_BW = torch.atan2(dy_BS, dx_BS); phi_BW = torch.acos(dz_BS / R_BW)
    k_tx = torch.sin(phi_BW) * torch.cos(theta_BW); k_tz = torch.cos(phi_BW)

    x1, z1 = xc - Lx/2, zc - Lz/2; x2, z2 = xc - Lx/2, zc + Lz/2
    x3, z3 = xc + Lx/2, zc - Lz/2; x4, z4 = xc + Lx/2, zc + Lz/2

    def ray_dir_py(xs, zs):
        dx = xs - x_BS; dy = torch.tensor(0.0 - y_BS, device=device); dz = zs - z_BS
        L = torch.sqrt(dx**2 + dy**2 + dz**2)
        return dx/L, dy/L, dz/L

    ux1, _, uz1 = ray_dir_py(x1, z1); ux2, _, uz2 = ray_dir_py(x2, z2)
    ux3, _, uz3 = ray_dir_py(x3, z3); ux4, _, uz4 = ray_dir_py(x4, z4)

    dx_WU = xu - x_BS; dy_WU = yu - y_BS; dz_WU = zu - z_BS
    L_USER = torch.sqrt(dx_WU**2 + dy_WU**2 + dz_WU**2)
    ux_user = dx_WU / L_USER; uz_user = dz_WU / L_USER

    ux_min = torch.min(torch.stack([ux1, ux2, ux3, ux4]))
    ux_max = torch.max(torch.stack([ux1, ux2, ux3, ux4]))
    uz_min = torch.min(torch.stack([uz1, uz2, uz3, uz4]))
    uz_max = torch.max(torch.stack([uz1, uz2, uz3, uz4]))

    beta_illum = 1000.0
    inx = torch.sigmoid(beta_illum * (ux_user - ux_min)) * torch.sigmoid(beta_illum * (ux_max - ux_user))
    inz = torch.sigmoid(beta_illum * (uz_user - uz_min)) * torch.sigmoid(beta_illum * (uz_max - uz_user))
    in_illumination = inx * inz

    dx_WU2 = xu - xc; dy_WU2 = yu; dz_WU2 = zu - zc
    R_WU = torch.sqrt(dx_WU2**2 + dy_WU2**2 + dz_WU2**2)
    t1 = dx_WU2 / R_WU; t2 = dz_WU2 / R_WU
    ax = (1.0 - in_illumination) * (k_tx - t1)
    az = (1.0 - in_illumination) * (k_tz - t2)
    sincx = torch.sinc((math.pi / lambda_val) * Lx * ax)
    sincz = torch.sinc((math.pi / lambda_val) * Lz * az)

    n_ant = torch.arange(E, dtype=torch.float32, device=device)

    # 路径 1 (LoS)
    phase1 = (2.0 * math.pi / lambda_val) * d_B * n_ant * torch.sin(theta_BW)
    a1 = (1.0 / math.sqrt(E)) * torch.complex(torch.cos(phase1), torch.sin(phase1))
    v1_mag = torch.tensor(lambda_val / (4.0 * math.pi), device=device) / R_BW
    v1_phase = torch.tensor(- (2.0 * math.pi / lambda_val), device=device) * R_BW
    v1 = torch.complex(v1_mag * torch.cos(v1_phase), v1_mag * torch.sin(v1_phase))
    H_path1 = v1 * a1.conj()

    # 路径 2 (NLoS)
    psi_2, tt_2, eta_2 = nlos_params
    phase2 = (2.0 * math.pi / lambda_val) * d_B * n_ant.unsqueeze(0) * torch.sin(tt_2).unsqueeze(1)
    a2 = (1.0 / math.sqrt(E)) * torch.complex(torch.cos(phase2), torch.sin(phase2))
    v2_mag = eta_2 * (lambda_val / (4.0 * math.pi * d_ref))
    v2 = torch.complex(v2_mag * torch.cos(psi_2), v2_mag * torch.sin(psi_2))
    H_path2 = v2.unsqueeze(1) * a2.conj()

    H_total = math.sqrt(E / L1) * (H_path1.unsqueeze(0) + H_path2)

    factor_mag = (Lx * Lz * sincx * sincz) / (lambda_val * R_WU)
    factor_phase = torch.tensor(- (2.0 * math.pi / lambda_val), device=device) * (k_tx * xc + k_tz * zc) - (math.pi / 2.0)
    factor = torch.complex(factor_mag * torch.cos(factor_phase), factor_mag * torch.sin(factor_phase))
    H_eq = H_total * factor.unsqueeze(1)
    return H_eq

# ==========================================
# MC 评估引擎
# ==========================================
def evaluate_window(window_params, N_MC, seed):
    """
    对给定窗口参数做 N_MC 组独立 Monte Carlo 评估
    每组：独立随机轨迹 + 独立 NLoS 路径参数
    返回：mean_outage, 全部 outage 列表
    """
    np.random.seed(seed)
    if not isinstance(window_params, torch.Tensor):
        window_params = torch.tensor(window_params, dtype=torch.float32, device=device)

    all_outages = []
    for mc in range(N_MC):
        traj = generate_human_trajectory()
        locs_t = torch.tensor(traj, dtype=torch.float32, device=device)

        psi_t  = torch.tensor(2 * np.pi * np.random.rand(N_TOTAL_POINTS), dtype=torch.float32, device=device)
        tt_t   = torch.tensor(-np.pi + 2 * np.pi * np.random.rand(N_TOTAL_POINTS), dtype=torch.float32, device=device)
        eta_t  = torch.tensor(10 ** ((-15 + 5 * np.random.rand(N_TOTAL_POINTS)) / 10), dtype=torch.float32, device=device)

        H_eq = equivalent_farfield_channel_pytorch(window_params, locs_t, (psi_t, tt_t, eta_t))
        H_w = torch.sum(H_eq, dim=1) / math.sqrt(E)
        dp = (torch.abs(H_w) ** 2) * p_m * P_BS
        intf = (n_users - 1) * dp
        sinr = dp / (intf + N0)
        rate = torch.log2(1.0 + sinr)
        outage = torch.mean((rate < R_th).float()).item()
        all_outages.append(outage)

    all_outages = np.array(all_outages)
    return float(np.mean(all_outages)), all_outages


# ============================================================
# 交叉验证：MATLAB 窗口 [5.09, 1.49, 0.10, 0.12]
# ============================================================
matlab_window = torch.tensor([5.09, 1.49, 0.10, 0.12], device=device)
area_matlab = 0.10 * 0.12

print("=" * 60)
print(" 交叉验证：Python 引擎独立评估 MATLAB GA 解")
print("=" * 60)
print(f" MATLAB 窗口: xc=5.09, zc=1.49, Lx=0.10, Lz=0.12")
print(f" MATLAB 声称: area=0.0120 m², outage=9.33%")
print(f" Python 评估: MC200 大样本统计")
print("-" * 60)

matlab_true_outage, all_outages = evaluate_window(matlab_window, N_MC=200, seed=99999)

print(f"\n MC200 均值 outage: {matlab_true_outage*100:.2f}%")
print(f" MC200 标准差:       {np.std(all_outages)*100:.2f}%")
print(f" MC200 最小值:       {np.min(all_outages)*100:.2f}%")
print(f" MC200 最大值:       {np.max(all_outages)*100:.2f}%")
print(f" MC200 < 10% 比例:   {np.mean(all_outages < 0.10)*100:.1f}%")

# 95% 置信区间
se = np.std(all_outages) / np.sqrt(len(all_outages))
ci_low = matlab_true_outage - 1.96 * se
ci_high = matlab_true_outage + 1.96 * se
print(f" 95% CI:             [{ci_low*100:.2f}%, {ci_high*100:.2f}%]")

print("\n" + "=" * 60)
if matlab_true_outage > 0.10:
    print(" 结论: MATLAB 解的真实中断率 > 10%，交叉验证不通过 ❌")
    print("       MATLAB 报告的 9.33% 是单样本运气值，非统计均值")
else:
    print(" 结论: MATLAB 解通过交叉验证 ✅")
print("=" * 60)

# ============================================================
# 对比：Python 最优解
# ============================================================
python_window = torch.tensor([4.742, 1.315, 0.157, 0.158], device=device)

print("\n" + "=" * 60)
print(" 对照：Python GA 最优解")
print("=" * 60)
print(f" Python 窗口: xc=4.74, zc=1.32, Lx=0.157, Lz=0.158")

python_true_outage, py_outages = evaluate_window(python_window, N_MC=200, seed=77777)

print(f" MC200 均值 outage: {python_true_outage*100:.2f}%")
print(f" MC200 标准差:       {np.std(py_outages)*100:.2f}%")

print("\n" + "=" * 60)
print(" 最终对比")
print("=" * 60)
print(f" {'方案':<15} {'面积(m²)':<12} {'MC200 outage':<15} {'可靠?'}")
print("-" * 60)
print(f" {'MATLAB GA':<15} {area_matlab:<12.4f} {matlab_true_outage*100:<15.2f}% {'❌ 单样本假象' if matlab_true_outage > 0.10 else '✅'}")
print(f" {'Python GA':<15} {0.157*0.158:<12.4f} {python_true_outage*100:<15.2f}% {'✅ 统计诚实' if python_true_outage < 0.105 else '⚠️'}")
print("=" * 60)